# Ranking Scorio Lite models with `scorio.rank`

This notebook compares the four model configurations on AIME 2026. Scorio ranking methods
take an `L x M x N` tensor: models by questions by sampled attempts.

|  |  |
| --- | --- |
| module | [`scorio/rank`](https://github.com/mohsenhariri/scorio/tree/main/scorio/rank) |
| method reference | [`scorio/rank/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/README.md) |
| paper | [Ranking Reasoning LLMs under Test-Time Scaling](https://aclanthology.org/2026.acl-long.1544/) |
| dataset | [Scorio Lite](https://huggingface.co/datasets/harimo/scorio-lite) |
| install | `pip install scorio` |

In [1]:
import pandas as pd
from datasets import load_dataset
from IPython.display import display

from scorio import rank

repo_name = "harimo/scorio-lite"
task = "aime_2026"

rows = (load_dataset(repo_name, "meta-math", split=task)
        .select_columns(["model_key", "data_id", "seed", "evalscope_is_correct"])
        .to_pandas()
        .sort_values(["model_key", "data_id", "seed"]))

models = sorted(rows.model_key.unique())
R = rows.evalscope_is_correct.to_numpy().astype(int).reshape(len(models), 30, 80)

print(R.shape, "models x questions x seeds")
print(models)

README.md: 0.00B [00:00, ?B/s]

meta-math/Qwen3.6-35B-A3B/aime_2026.parq(…):   0%|          | 0.00/39.9M [00:00<?, ?B/s]

meta-math/gpt-oss-20b_high/aime_2026.par(…):   0%|          | 0.00/19.6M [00:00<?, ?B/s]

meta-math/gpt-oss-20b_low/aime_2026.parq(…):   0%|          | 0.00/1.67M [00:00<?, ?B/s]

meta-math/gpt-oss-20b_medium/aime_2026.p(…):   0%|          | 0.00/2.47M [00:00<?, ?B/s]

...

Generating cmimc_2025 split:   0%|          | 0/12800 [00:00<?, ? examples/s]

Generating hmmt_feb_2026 split:   0%|          | 0/10560 [00:00<?, ? examples/s]

Generating hmmt_nov_2025 split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating smt_2025 split:   0%|          | 0/16960 [00:00<?, ? examples/s]

(4, 30, 80) models x questions x seeds
['Qwen3.6-35B-A3B', 'gpt-oss-20b_high', 'gpt-oss-20b_low', 'gpt-oss-20b_medium']


## Bayes@N ranking

Rank 1 is the best model. Asking for scores as well shows the values used to produce the
order.

In [2]:
ranks, scores = rank.bayes(R, return_scores=True)

leaderboard = pd.DataFrame(
    {"rank": ranks.astype(int), "Bayes@N": scores},
    index=models,
)

display(leaderboard.sort_values("rank").round(3))

,rank,Bayes@N
Qwen3.6-35B-A3B,1,0.912
gpt-oss-20b_high,2,0.867
gpt-oss-20b_medium,3,0.770
gpt-oss-20b_low,4,0.427


## Comparing ranking methods

The methods below use different summaries of the same tensor. `rasch_mml` is used
instead of plain `rasch` because this split contains boundary questions that every model
gets right or wrong; marginal maximum likelihood handles those questions explicitly.

In [3]:
methods = ["bayes", "borda", "win_rate", "bradley_terry",
           "elo", "rasch_mml", "thompson"]

comparison = pd.DataFrame(
    {method: getattr(rank, method)(R).astype(int) for method in methods},
    index=models,
).sort_values("bayes")

display(comparison)

,bayes,borda,win_rate,bradley_terry,elo,rasch_mml,thompson
Qwen3.6-35B-A3B,1,1,1,1,1,1,1
gpt-oss-20b_high,2,2,2,2,2,2,2
gpt-oss-20b_medium,3,3,3,3,3,3,3
gpt-oss-20b_low,4,4,4,4,4,4,4


## Ranking at different sample budgets

The order at one seed differs from the order at 80 seeds: the medium and high gpt-oss
configurations exchange places. From two samples onward, this sweep matches the 80-sample
order. The reported result should still include the sample budget.

In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame(
    {f"n={n}": rank.bayes(R[:, :, :n]).astype(int) for n in budgets},
    index=models,
).sort_values("n=80")

display(sweep)
print("models whose rank changes between n=1 and n=80:",
      int((sweep["n=1"] != sweep["n=80"]).sum()), "of", len(models))

,n=1,n=2,n=4,n=8,n=16,n=32,n=80
Qwen3.6-35B-A3B,1,1,1,1,1,1,1
gpt-oss-20b_high,3,2,2,2,2,2,2
gpt-oss-20b_medium,2,3,3,3,3,3,3
gpt-oss-20b_low,4,4,4,4,4,4,4


models whose rank changes between n=1 and n=80: 2 of 4


## Related ranking methods

Scorio includes voting, paired-comparison, item-response, graph, and listwise ranking
methods. The [method reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/README.md)
lists the assumptions and source for each one.